# Pre-processing of ESCO data
Felix Zaussinger | 14.02.2022

## Core Analysis Goal(s)
1. Calculate occupation-skills and occupation similarity matrices based on ESCO v.1.1.0
    - occupation-skills matrix: unweighted, essential/optional
    - occupation-similarity matrix: co-occurrence (optional: cosine similarity, relatedness, etc.)
2. Configure static parameters in config file
3. Create skills metadata file (green/non-green, coreness)

## Key Insight(s)
1.
2.
3.

In [1]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
from tqdm import tqdm

from src import utils

%load_ext autoreload
%autoreload 2

import pandas as pd
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

Define directory structure

In [2]:
# project directory
abspath = os.path.abspath('')
project_dir = str(Path(abspath).parents[0])

# sub-directories
data_raw = os.path.join(project_dir, "data", "raw")
data_interim = os.path.join(project_dir, "data", "interim")
data_processed = os.path.join(project_dir, "data", "processed")
figure_dir = os.path.join(project_dir, "reports", "figures")

Import config file

In [3]:
# load
config = utils.load_config(os.path.join(project_dir, "configs", "main_config.yml"))
config

{'EU_LFS': {'FPATH_RAW': 'r"C:\\eurostat_data\\raw"',
  'FPATH_RAW_YF': 'C:\\eurostat_data\\raw\\Yearly_Data\\YearlyFiles_83_2019\\_YearlyFiles',
  'FPATH_INTERIM': 'C:\\eurostat_data\\interim',
  'FPATH_PROCESSED': 'r"C:\\eurostat_data\\processed"'},
 'ESCO': {'LANGUAGE': 'en',
  'VERSION': 'v1.1.0',
  'WEIGHT_UNIFORM': 1,
  'WEIGHT_ESSENTIAL_SKILL': 1,
  'WEIGHT_OPTIONAL_SKILL': 0.5},
 'TRANSITION_ANALYSIS': {'MIN_VIABLE': 0.3,
  'HIGHLY_VIABLE': 0.4,
  'MAX_JOB_ZONE_DIF': 1,
  'MIN_EARNINGS_RATIO': 0.75}}

#### Load ESCO data

In [4]:
# esco configurations
esco_language = config["ESCO"]["LANGUAGE"] # "en"
esco_version = config["ESCO"]["VERSION"] # "v1.0.3" or "v1.1.0"
esco_skills_hierarchy_version = "v1.0.8" if esco_version == "v1.0.3" else "v1.1.0"

# core
occ = pd.read_csv(os.path.join(data_raw, "esco", esco_version, "occupations_{}.csv".format(esco_language)))
skills = pd.read_csv(os.path.join(data_raw, "esco", esco_version, "skills_{}.csv".format(esco_language)))
occ_skills_mapping = pd.read_csv(os.path.join(data_raw, "esco", esco_version, "occupationSkillRelations.csv"))

# additional
skill_groups = pd.read_csv(os.path.join(data_raw, "esco", esco_version, "skillGroups_{}.csv".format(esco_language)))
skills_hierarchy = pd.read_csv(os.path.join(data_raw, "esco", esco_skills_hierarchy_version, "skillsHierarchy_{}.csv".format(esco_language)))

skills_hierarchy_kanders = pd.read_csv(os.path.join(data_raw, "mapping-career-causeways", "codebase", "data", "processed", "ESCO_skills_hierarchy", "ESCO_skills_hierarchy.csv"))

### Build occupation-skills matrix

**Encoding**
- 0: skill not required
- 1: skill required, essential
- 2: skill required, optional

In [5]:
%%time

target_path = os.path.join(project_dir, "data", "interim", "esco", esco_version, "occ_skills_matrix_weighted.pkl")

if not os.path.exists(target_path):
    errors = 0
    skill_vectors = []

    for i in tqdm(range(len(occ))):
        occ_uri = occ.iloc[i, :][1]

        # lookup corresponding skills
        skill_list = occ_skills_mapping[occ_skills_mapping["occupationUri"] == occ_uri]

        # create vector
        skill_vector = []
        for j, skill in enumerate(skills.conceptUri.values):

            if skill in skill_list.skillUri.values:
                relation_type = skill_list.loc[skill_list.skillUri == skill, "relationType"].values[0]

                # skill needed for occupation and essential
                if relation_type == "essential":
                    skill_vector.append(1)
                # skill needed for occupation and optional
                elif relation_type == "optional":
                    skill_vector.append(2)
            else:
                # skill not needed for occupation
                skill_vector.append(0)

        indices = [i for i, j in enumerate(skill_vector) if j == 1]

        # sanity check
        if len(skill_list.skillUri) != np.sum(np.invert(np.array(skill_vector) == 0)):
            errors += 1

        # append
        skill_vectors.append(skill_vector)

    # info
    print("n_errors: ", errors)

    # create df
    occ_skills_matrix_eo = pd.DataFrame(
        index=occ.conceptUri,
        columns=skills.conceptUri,
        data=np.array(skill_vectors)
    )

    # save to disk
    occ_skills_matrix_eo.to_pickle(target_path)
else:
    # read from disk
    occ_skills_matrix_eo = pd.read_pickle(target_path)

CPU times: total: 203 ms
Wall time: 401 ms


Calculate weighted and unweighted variants

In [6]:
# weighted form
replace_weighted = {
    1: config["ESCO"]["WEIGHT_ESSENTIAL_SKILL"],
    2: config["ESCO"]["WEIGHT_OPTIONAL_SKILL"]
}
occ_skills_matrix_weighted = occ_skills_matrix_eo.replace(to_replace=replace_weighted)

# unweighted form
occ_skills_matrix_unweighted = occ_skills_matrix_eo.replace(to_replace=[1, 2], value=config["ESCO"]["WEIGHT_UNIFORM"])

#### Disaggregation step (ESCO -> ISCO-08 -> KldB)
TODO

#### Calculate (co-occurrence) occupation similarity matrix

Weighted form

In [7]:
%%time

target_path = os.path.join(project_dir, "data", "interim", "esco", esco_version, "occ_sim_matrix_weighted_coo.pkl")

if not os.path.exists(target_path):

    # calculate co-occurrence matrix via matrix multiplication with transpose form
    occ_sim_matrix_weighted_coo = np.dot(
        occ_skills_matrix_weighted.values,
        occ_skills_matrix_weighted.values.transpose()
    )

    # to df
    df_occ_sim_matrix_weighted_coo = pd.DataFrame(
        index=occ.conceptUri,
        columns=occ.conceptUri,
        data=occ_sim_matrix_weighted_coo
    )

    # save
    df_occ_sim_matrix_weighted_coo.to_pickle(target_path)
else:
    # read
    df_occ_sim_matrix_weighted_coo = pd.read_pickle(target_path)

CPU times: total: 31.2 ms
Wall time: 80.9 ms


Unweighted form

%%time

target_path = os.path.join(project_dir, "data", "interim", "esco", esco_version, "occ_sim_matrix_unweighted_coo.pkl")

if not os.path.exists(target_path):

    # calculate co-occurrence matrix via matrix multiplication with transpose form
    occ_sim_matrix_unweighted_coo = np.dot(
        occ_skills_matrix_unweighted.values,
        occ_skills_matrix_unweighted.values.transpose()
    )

    # to df
    df_occ_sim_matrix_unweighted_coo = pd.DataFrame(
        index=occ.conceptUri,
        columns=occ.conceptUri,
        data=occ_sim_matrix_unweighted_coo
    )

    # save
    df_occ_sim_matrix_unweighted_coo.to_pickle(target_path)
else:
    # read
    df_occ_sim_matrix_unweighted_coo = pd.read_pickle(target_path)

#### Create ESCO Skills Metadata File

Green Skills

In [8]:
# read green skill data
green_skills = pd.read_csv(os.path.join(data_raw, "esco", esco_version, "greenSkillsCollection_{}.csv".format(esco_language)))
green_id_colname = "skillGreen"
green_skills[green_id_colname] = True

In [9]:
# find cols that are unique in green skills file compared to general skills file
# https://www.kaggle.com/ashukr/sets-and-venn-diagram-in-python: The difference between A and B contains all elements that are in A but not in B
set_diff = list(set(green_skills.columns.values.tolist()) - set(skills.columns.values.tolist()))
set_diff.insert(0, "conceptUri")

In [10]:
# copy skills df and join information on green skills
skills_metadata = skills.copy()
skills_metadata = skills_metadata.merge(right=green_skills[set_diff], on="conceptUri", how="left", validate="one_to_one")
skills_metadata = skills_metadata.fillna(value={green_id_colname: False})

Coreness

In [11]:
skills_coreness = pd.read_csv(
    os.path.join(data_raw, "mapping-career-causeways", "codebase", "data", "interim", "upskilling_analysis", "skills_coreness_measure.csv")
)

# append conceptUri of ESCO v1.0.3 skills
skills_v103 = pd.read_csv(os.path.join(data_raw, "esco", "v1.0.3", "skills_{}.csv".format(esco_language)))
keep_cols = ["conceptUri", "preferredLabel"]
skills_coreness = skills_coreness.merge(right=skills_v103[keep_cols], left_on="preferred_label", right_on="preferredLabel", how="left", validate="one_to_one")

In [12]:
keep_cols = ["conceptUri", "coreness"]
skills_metadata = skills_metadata.merge(right=skills_coreness[keep_cols], on="conceptUri", how="left", validate="one_to_one")

**Evaluation**

- ESCO v.1.1.0 comprises 13891 skills compared to v.1.0.3 with 13485 skills.
- Hence, overall 406 new skills in v.1.1.0 compared to v.1.0.3.
- 570 skills labelled as green in v.1.1.0, some of which are presumably new, while other's already existed.
- After merging we don't obtain coreness values for 512 skills in v.1.1.0.
- This means that the conceptUri of 512 - 406 = 106 skills that existed in the last version must have changed.

In [13]:
print(len(skills))  # new esco
print(len(skills_coreness))  # old esco
print(len(skills) - len(skills_coreness))
print(len(skills_metadata) - len(skills_metadata.coreness.dropna()))  # (probably) existing in v1.1.0 but unmatched

13891
13485
406
512


In [14]:
# save: dataset refers to variable"greenskill" in the FDZ Verfahrensbeschreibung
skills_metadata.to_csv(
    os.path.join(project_dir, "data", "interim", "esco", esco_version, "skills_metadata_{}.csv".format(esco_language))
)

#### Greenness scores

TODO
- merge Vona et al. Greenness/Brownness to occupation metadata file
- compare with ESCO greenness
- merge other onet metadata (job zone and constituent variables)

Unweighted (no consideration of essential/optional skills)

In [15]:
# number of occupation-specific skills
n_total_specific_skills = occ_skills_matrix_unweighted.sum(axis=1).values

# number of occupation-specific green skills
green_specific_skills = occ_skills_matrix_unweighted.values * skills_metadata.skillGreen.astype(np.int8).values
n_green_specific_skills = green_specific_skills.sum(axis=1)

# greenness
greenness_esco = n_green_specific_skills / n_total_specific_skills

# create df
data = {
    "n_total_specific_skills": n_total_specific_skills,
    "n_green_specific_skills": n_green_specific_skills,
    "greenness_esco": greenness_esco
}

df_greenness_esco = pd.DataFrame(index=occ_skills_matrix_unweighted.index, data=data)
df_greenness_esco = df_greenness_esco.reset_index()

Weighted (consideration of essential/optional skills)

In [16]:
# TODO

Read data from Vona et. al 2018 and Green Task Project (GTP)

In [17]:
greenness_onet_gtp = pd.read_excel(
    io=os.path.join(data_raw, "onet", "green_task_project", "Onet_GreenTask_AppA.xlsx"),
    sheet_name="Occupations"
)

greenness_onet_vona = pd.read_excel(
    io=os.path.join(data_raw, "onet", "vona_2018", "vona_2018_table_a1.xlsx"),
    sheet_name="Greenness"
)

# merge
greenness_onet = greenness_onet_gtp.copy()
greenness_onet = greenness_onet.merge(right=greenness_onet_vona, on="onet_code", how="left", validate="1:1", suffixes=["_gtp", "_vona2018"])

# save
greenness_onet.to_csv(
    os.path.join(project_dir, "data", "interim", "onet", "task_based_greenness_onet.csv")
)

In [18]:
import pingouin as pg
greenness_onet[["greenness_gtp", "greenness_vona2018"]].rcorr(method="pearson")

,greenness_gtp,greenness_vona2018
greenness_gtp,-,***
greenness_vona2018,0.997,-


C:\Users\fzaussinger\Miniconda3\envs\re4gt\lib\site-packages\outdated\utils.py:14: OutdatedPackageWarning: The package pingouin is out of date. Your version is 0.5.0, the latest is 0.5.1.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  return warn(


**Merge O*NET data to ESCO**
TODO: figure out difference with latest version (Nov2020). later use this one and inherit codes to lower level esco occs.

In [19]:
# using this crosswalk we may miss data for the 68 new occupations
onet_esco_crosswalk = pd.read_csv(
    os.path.join(data_raw, "mapping-career-causeways", "codebase", "data", "processed", "ESCO_ONET_xwalk_full.csv")
)

# merge data from crosswalk
greenness_onet_esco = greenness_onet.copy()
greenness_onet_esco = greenness_onet_esco.merge(right=onet_esco_crosswalk, on="onet_code", how="left", validate="1:m")

# save
greenness_onet_esco.to_csv(
    os.path.join(project_dir, "data", "interim", "onet", "task_based_greenness_esco.csv")
)

In [20]:
# merge ESCO and O*NET greenness scores to occupation df
occ_metadata = occ.copy()

# ESCO-based
occ_metadata = occ_metadata.merge(right=df_greenness_esco, on="conceptUri", how="left", validate="one_to_one")

# O*NET-based
occ_metadata = occ_metadata.merge(right=greenness_onet_esco, left_on="conceptUri", right_on="concept_uri", how="left", validate="1:m")

In [21]:
occ_metadata[["greenness_gtp", "greenness_vona2018", "greenness_esco"]].rcorr(method="pearson", upper="pval")

,greenness_gtp,greenness_vona2018,greenness_esco
greenness_gtp,-,***,***
greenness_vona2018,0.995,-,***
greenness_esco,0.743,0.761,-


In [22]:
occ_metadata[["greenness_gtp", "greenness_vona2018", "greenness_esco"]].rcorr(method="pearson", upper="n")

,greenness_gtp,greenness_vona2018,greenness_esco
greenness_gtp,-,415,431
greenness_vona2018,0.995,-,415
greenness_esco,0.743,0.761,-


In [23]:
occ_metadata.to_csv(
    os.path.join(project_dir, "data", "interim", "esco", esco_version, "occ_metadata_{}.csv".format(esco_language))
)

In [24]:
occ_metadata.sort_values("greenness_esco", ascending=False).head(30)

,conceptType,conceptUri,iscoGroup,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,regulatedProfessionNote,scopeNote,definition,inScheme,description,code,n_total_specific_skills,n_green_specific_skills,greenness_esco,onet_code,title_gtp,occupation_type,n_new_green_tasks_gtp,n_existing_green_tasks_gtp,n_non_green_tasks_gtp,greenness_gtp,title_vona2018,greenness_vona2018,total_spec_tasks_vona2018,green_spec_tasks_vona2018,id,concept_uri,preferred_label,isco_level_4,onet_occupation
2417,Occupation,http://data.europa.eu/esco/occupation/c7f5cab7...,3112,energy assessor,energy performance certificate assessor\nenerg...,NaN,released,2016-07-05T16:04:20Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Energy assessors determine the energy performa...,3112.1.5,22,20,0.909091,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2946,Occupation,http://data.europa.eu/esco/occupation/fab474ea...,2133,natural resources consultant,resources consultant\nnatural resources expert...,NaN,released,2017-01-17T14:27:38Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Natural resources consultant provide advice on...,2133.8,33,26,0.787879,19-2041.03,Industrial Ecologists,New Green N&E,38.0,0.0,0.0,1.000000,NaN,NaN,NaN,NaN,2880.0,http://data.europa.eu/esco/occupation/fab474ea...,natural resources consultant,2133.0,industrial ecologists
443,Occupation,http://data.europa.eu/esco/occupation/23a61ff1...,3112,energy conservation officer,energy information officer\nenergy management ...,NaN,released,2020-12-09T08:38:48.962Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Energy conservation officers promote the conse...,3112.6,16,12,0.750000,47-4099.03,Weatherization Installers and Technicians,New Green N&E,18.0,0.0,0.0,1.000000,Weatherization Installers and Technicians,1.000000,18.0,18.0,430.0,http://data.europa.eu/esco/occupation/23a61ff1...,energy conservation officer,3112.0,weatherization installers and technicians
2195,Occupation,http://data.europa.eu/esco/occupation/b5d2db88...,2422,environmental policy officer,environmental officer\nenvironmental policy ad...,NaN,released,2021-12-08T20:02:08.2Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,"Environmental policy officers research, analys...",2422.12.5,36,27,0.750000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2468,Occupation,http://data.europa.eu/esco/occupation/cbde1a3a...,3112,energy analyst,energy procurement analyst\nenergy performanc...,NaN,released,2017-01-17T14:44:17Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Energy analysts evaluate the consumption of en...,3112.5,24,17,0.708333,13-1199.01,Energy Auditors,Existing Green N&E,4.0,17.0,0.0,1.000000,Energy Auditors,1.000000,21.0,21.0,2412.0,http://data.europa.eu/esco/occupation/cbde1a3a...,energy analyst,3112.0,energy auditors
2021,Occupation,http://data.europa.eu/esco/occupation/a7358961...,2143,environmental expert,environmental scientist\nsoil scientist\nenvir...,NaN,released,2020-12-08T12:24:26.113Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Environmental experts search for technological...,2143.2,34,24,0.705882,17-2081.00,Environmental Engineers,Green Enhanced Skills,3.0,25.0,0.0,1.000000,Environmental Engineers,1.000000,28.0,28.0,1976.0,http://data.europa.eu/esco/occupation/a7358961...,environmental expert,2143.0,environmental engineers
2654,Occupation,http://data.europa.eu/esco/occupation/de0856fe...,3257,hazardous waste inspector,hazardous waste management inspector\nhazardou...,NaN,released,2016-12-09T12:00:32Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Hazardous waste